# log-back — worked example 2: Compose log_back through z = log(x + y)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For `z = log(x + y)`, hand-composing back fns gives the closed form `dL/dx = dL/dy = 1/(x+y)` when the loss is `z.sum()`. The add op routes the same upstream gradient to both parents unchanged, then `log_back` divides by the sum.

## Worked solution

We chain two back fns and confirm the algebra simplifies.

1. **Forward.** Compute `s = x + y` and `z = log(s)`, with loss `L = z.sum()`, so the seed is `dL/dz = ones_like(z)`.
2. **Back through log.** `dL/ds = log_back(dL/dz, z, s) = ones / s = 1/(x+y)`.
3. **Back through add.** Addition has derivative 1 w.r.t. each operand, so `add_back0` and `add_back1` both just pass `dL/ds` through unchanged. Hence `dL/dx = dL/dy = 1/(x+y)`.
4. **Witness.** We verify both gradients against `torch.autograd`.

The demo prints whether both composed gradients match autograd.

In [ ]:
import torch as t

t.manual_seed(1)

def log_back(grad_out, out, x):
    return grad_out / x

def add_back0(grad_out, out, x, y):
    return grad_out          # d(x+y)/dx = 1

def add_back1(grad_out, out, x, y):
    return grad_out          # d(x+y)/dy = 1

def compose_log_add_back(x, y):
    s = x + y
    z = t.log(s)
    dL_dz = t.ones_like(z)
    dL_ds = log_back(dL_dz, z, s)
    dL_dx = add_back0(dL_ds, s, x, y)
    dL_dy = add_back1(dL_ds, s, x, y)
    return dL_dx, dL_dy

x = t.rand(4) + 0.2
y = t.rand(4) + 0.2
dx, dy = compose_log_add_back(x, y)

xv = x.clone().detach().requires_grad_(True)
yv = y.clone().detach().requires_grad_(True)
t.log(xv + yv).sum().backward()
print('dx matches:', t.allclose(dx, xv.grad))
print('dy matches:', t.allclose(dy, yv.grad))